# OptiPricer — NIFTY Options Analysis

This notebook demonstrates a complete workflow for pricing and analyzing NIFTY index options using OptiPricer.

**Topics covered:**
1. Pricing single options with the facade API
2. First & second-order Greeks analysis
3. Building an option chain
4. Constructing and visualizing a volatility surface/smile
5. Analyzing an Iron Condor strategy with full Greeks and payoff plots

In [ ]:
import optipricer
from optipricer.models import BlackScholesModel, GreeksCalculator
from optipricer.strategies import IronCondor, BullCallSpread, LongStraddle
from optipricer.chain import OptionChain, generate_nifty_strikes
from optipricer.surface import VolatilitySurface
import optipricer.viz
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

## 1. Pricing NIFTY Options

Let's price a NIFTY 50 call and put option using typical market parameters.

In [ ]:
# NIFTY Market Parameters (example)
NIFTY_SPOT = 24500.0      # Current NIFTY spot price
RISK_FREE = 0.07          # India 10Y govt bond yield (~7%)
DIVIDEND_YIELD = 0.012    # NIFTY dividend yield (~1.2%)
DAYS_TO_EXPIRY = 30       # Monthly expiry
T = DAYS_TO_EXPIRY / 365.0
IV = 0.14                 # Near-ATM implied volatility (~14%)

# Strike selection (ATM ± 200 points)
K_ATM = 24500.0
K_OTM_CALL = 24700.0
K_OTM_PUT = 24300.0

print("=" * 60)
print(f"{'NIFTY Options Pricing':^60}")
print("=" * 60)
print(f"Spot: {NIFTY_SPOT:,.0f} | DTE: {DAYS_TO_EXPIRY} | IV: {IV*100:.1f}%")
print(f"Risk-Free: {RISK_FREE*100:.1f}% | Div Yield: {DIVIDEND_YIELD*100:.1f}%")
print("-" * 60)

for strike in [K_OTM_PUT, K_ATM, K_OTM_CALL]:
    call = optipricer.price(S=NIFTY_SPOT, K=strike, r=RISK_FREE, T=T, vol=IV, q=DIVIDEND_YIELD, option='call')
    put = optipricer.price(S=NIFTY_SPOT, K=strike, r=RISK_FREE, T=T, vol=IV, q=DIVIDEND_YIELD, option='put')
    moneyness = 'ATM' if strike == K_ATM else ('OTM Call' if strike > NIFTY_SPOT else 'OTM Put')
    print(f"K={strike:>8,.0f} ({moneyness:>8}) | Call: ₹{call:>8.2f} | Put: ₹{put:>8.2f}")

print("=" * 60)

## 2. Complete Greeks Analysis

OptiPricer computes both first-order and second-order (Vanna, Volga, Charm) Greeks.

In [ ]:
# Compute all Greeks for ATM option
g = optipricer.greeks(S=NIFTY_SPOT, K=K_ATM, r=RISK_FREE, T=T, vol=IV, q=DIVIDEND_YIELD)

print("=" * 60)
print(f"{'Greeks for NIFTY ATM (K=24500)':^60}")
print("=" * 60)
print(f"\n{'First-Order Greeks':}")
print(f"  Call Delta:  {g['call_delta']:>+.4f}")
print(f"  Put Delta:   {g['put_delta']:>+.4f}")
print(f"  Gamma:       {g['gamma']:>.6f}")
print(f"  Vega:        {g['vega']:>.4f} (per 1% vol change)")
print(f"  Call Theta:  {g['call_theta']:>+.4f} (₹/day)")
print(f"  Put Theta:   {g['put_theta']:>+.4f} (₹/day)")
print(f"  Call Rho:    {g['call_rho']:>+.4f}")
print(f"  Put Rho:     {g['put_rho']:>+.4f}")
print(f"\n{'Second-Order Greeks':}")
print(f"  Vanna:       {g['vanna']:>+.6f} (dDelta/dVol)")
print(f"  Volga:       {g['volga']:>+.4f} (dVega/dVol)")
print(f"  Call Charm:  {g['call_charm']:>+.6f} (dDelta/day)")
print(f"  Put Charm:   {g['put_charm']:>+.6f} (dDelta/day)")
print("=" * 60)

In [ ]:
# Plot Delta and Gamma vs Spot Price
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

spot_range = np.linspace(23500, 25500, 200)
call_deltas = []
gammas = []
vannas = []

for s in spot_range:
    model = BlackScholesModel(strike_price=K_ATM, volatility=IV, risk_free_rate=RISK_FREE,
                              time_to_maturity=T, underlying_price=s, dividend_yield=DIVIDEND_YIELD)
    calc = GreeksCalculator(model)
    call_deltas.append(calc.call_delta())
    gammas.append(calc.gamma())
    vannas.append(calc.vanna())

axes[0].plot(spot_range, call_deltas, color='#2196F3', linewidth=2.5, label='Call Delta')
axes[0].axhline(0.5, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(K_ATM, color='#E91E63', linestyle='--', alpha=0.5, label=f'ATM K={K_ATM}')
axes[0].set_title('Call Delta vs Spot Price', fontsize=13, fontweight='bold')
axes[0].set_xlabel('NIFTY Spot Price')
axes[0].set_ylabel('Delta (Δ)')
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].plot(spot_range, gammas, color='#FF9800', linewidth=2.5, label='Gamma')
axes[1].axvline(K_ATM, color='#E91E63', linestyle='--', alpha=0.5, label=f'ATM K={K_ATM}')
axes[1].set_title('Gamma vs Spot Price', fontsize=13, fontweight='bold')
axes[1].set_xlabel('NIFTY Spot Price')
axes[1].set_ylabel('Gamma (Γ)')
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.suptitle(f'NIFTY Greeks Sensitivity (K={K_ATM}, {DAYS_TO_EXPIRY} DTE)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Building an Option Chain

Generate a broker-terminal-style chain across multiple strikes, similar to what you'd see on NSE or Zerodha.

In [ ]:
# Generate NIFTY strikes centered on spot
strikes = generate_nifty_strikes(NIFTY_SPOT, step=100.0, count=11)
print(f"Strike range: {strikes[0]:.0f} to {strikes[-1]:.0f} (step=100)")

# Build the option chain
chain = OptionChain(
    S=NIFTY_SPOT, r=RISK_FREE, T=T,
    strikes=strikes, vol=IV, q=DIVIDEND_YIELD
)
data = chain.build()

# Pretty-print the chain
print(f"\n{'NIFTY Option Chain — {0} DTE'.format(DAYS_TO_EXPIRY)}")
print("=" * 90)
print(f"{'Strike':>8} | {'Call':>8} {'Put':>8} | {'Δ(C)':>7} {'Δ(P)':>7} | {'Γ':>8} {'ν':>7} | {'Θ(C)':>7}")
print("-" * 90)
for row in data:
    marker = ' ◀ ATM' if row['strike'] == chain.atm_strike() else ''
    print(f"{row['strike']:>8.0f} | {row['call_price']:>8.2f} {row['put_price']:>8.2f} | "
          f"{row['call_delta']:>+7.4f} {row['put_delta']:>+7.4f} | "
          f"{row['gamma']:>8.6f} {row['vega']:>7.4f} | "
          f"{row['call_theta']:>+7.4f}{marker}")
print("=" * 90)

# Summary
summary = chain.summary()
print(f"\nATM Strike: {summary['atm_strike']:.0f} | ATM Call: ₹{summary['atm_call_price']:.2f} | ATM Put: ₹{summary['atm_put_price']:.2f}")

## 4. Volatility Surface & Smile

Construct a volatility surface from multiple expiry slices and visualize the skew/smile.

In [ ]:
# Simulated NIFTY IV data across strikes and expiries
# In practice, you'd get this from NSE option chain snapshots
surface_strikes = [24000, 24100, 24200, 24300, 24400, 24500, 24600, 24700, 24800, 24900, 25000]

# IV data for different expiries (realistic NIFTY smile shape)
chain_data = [
    {
        'expiry': 7/365,   # Weekly expiry
        'strikes': surface_strikes,
        'ivs': [0.175, 0.168, 0.160, 0.155, 0.148, 0.145, 0.148, 0.155, 0.160, 0.168, 0.175]
    },
    {
        'expiry': 14/365,  # Bi-weekly
        'strikes': surface_strikes,
        'ivs': [0.170, 0.163, 0.156, 0.150, 0.145, 0.142, 0.145, 0.150, 0.156, 0.163, 0.170]
    },
    {
        'expiry': 30/365,  # Monthly expiry
        'strikes': surface_strikes,
        'ivs': [0.165, 0.158, 0.152, 0.147, 0.143, 0.140, 0.143, 0.147, 0.152, 0.158, 0.165]
    },
    {
        'expiry': 60/365,  # Next month
        'strikes': surface_strikes,
        'ivs': [0.160, 0.155, 0.150, 0.145, 0.142, 0.140, 0.142, 0.145, 0.150, 0.155, 0.160]
    },
]

# Construct the surface
surface = VolatilitySurface.from_chain_data(chain_data)
print(surface)

# Interpolate IV at arbitrary points
test_points = [(24350, 10/365), (24550, 20/365), (24500, 45/365)]
print("\nInterpolated IVs:")
for k, t in test_points:
    iv = surface.get_iv(k, t)
    print(f"  K={k}, {t*365:.0f} DTE → IV = {iv*100:.2f}%")

In [ ]:
# Plot volatility smile for nearest expiry
fig = surface.plot_smile(expiry_index=0, title='NIFTY Weekly Volatility Smile')
plt.show()

In [ ]:
# Plot 3D volatility surface
fig = surface.plot(title='NIFTY Implied Volatility Surface')
plt.show()

## 5. Iron Condor Strategy Analysis

Analyze an Iron Condor on NIFTY with full portfolio-level Greeks and payoff visualization.

In [ ]:
# Define an Iron Condor
ic = IronCondor(
    S=NIFTY_SPOT,
    sigma=IV,
    r=RISK_FREE,
    T=T,
    put_strike_long=24000,
    put_strike_short=24200,
    call_strike_short=24800,
    call_strike_long=25000,
    q=DIVIDEND_YIELD
)

# Full portfolio Greeks
print("=" * 60)
print(f"{'NIFTY Iron Condor Analysis':^60}")
print("=" * 60)
print(f"Legs: Buy P{24000} / Sell P{24200} / Sell C{24800} / Buy C{25000}")
print(f"DTE: {DAYS_TO_EXPIRY} | Spot: {NIFTY_SPOT:,.0f}")
print("-" * 60)
print(f"\n  Strategy Value: ₹{ic.total_value():>+.2f}")
print(f"  Net Credit:     ₹{-ic.total_value():>.2f} (per unit)")
print(f"  NIFTY Lot (50): ₹{-ic.total_value() * 50:>,.2f}")
print(f"\n{'Portfolio Greeks':}")
print(f"  Delta:  {ic.total_delta():>+.6f}")
print(f"  Gamma:  {ic.total_gamma():>+.6f}")
print(f"  Vega:   {ic.total_vega():>+.6f}")
print(f"  Theta:  {ic.total_theta():>+.6f} (₹/day)")
print(f"  Rho:    {ic.total_rho():>+.6f}")
print(f"  Vanna:  {ic.total_vanna():>+.6f}")
print(f"  Volga:  {ic.total_volga():>+.6f}")
print(f"  Charm:  {ic.total_charm():>+.8f}")

# Calculate P&L scenarios
credit = -ic.total_value()
max_loss = ic.payoff_at_expiration(23000) - ic.total_value()  # Beyond wings
print(f"\n{'P&L Scenarios':}")
print(f"  Max Profit (range-bound):  ₹{credit:>+.2f}")
print(f"  Max Loss (beyond wings):   ₹{max_loss:>+.2f}")
print(f"  Risk/Reward Ratio:         {abs(max_loss/credit):.2f}:1")
print("=" * 60)

In [ ]:
# Plot the Iron Condor payoff profile
fig = optipricer.viz.plot_payoff(ic, title='NIFTY Iron Condor — Payoff at Expiry')
plt.show()

In [ ]:
# Compare multiple strategies side by side
strategies = [
    ('Bull Call Spread', BullCallSpread(NIFTY_SPOT, IV, RISK_FREE, T, 24400, 24600, q=DIVIDEND_YIELD)),
    ('Long Straddle', LongStraddle(NIFTY_SPOT, IV, RISK_FREE, T, 24500, q=DIVIDEND_YIELD)),
    ('Iron Condor', ic),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, strat) in zip(axes, strategies):
    positions = strat.get_positions()
    strikes_list = [p.strike for p in positions]
    s_min = max(0, min(strikes_list) * 0.97)
    s_max = max(strikes_list) * 1.03
    spot_range = np.linspace(s_min, s_max, 200)
    
    payoffs = [strat.payoff_at_expiration(s) for s in spot_range]
    net_pnl = [p - strat.total_value() for p in payoffs]
    
    ax.plot(spot_range, net_pnl, color='#1f77b4', linewidth=2.5)
    ax.fill_between(spot_range, net_pnl, 0, where=[p > 0 for p in net_pnl], 
                    facecolor='#2ecc71', alpha=0.2)
    ax.fill_between(spot_range, net_pnl, 0, where=[p < 0 for p in net_pnl], 
                    facecolor='#e74c3c', alpha=0.2)
    ax.axhline(0, color='gray', linestyle='-', linewidth=1)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('NIFTY at Expiry')
    ax.set_ylabel('P&L (₹)')
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(f'Strategy Comparison ({DAYS_TO_EXPIRY} DTE)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

- **Option pricing** using the high-level facade API and the advanced OO API
- **Complete Greeks analysis** including second-order Greeks (Vanna, Volga, Charm)
- **Option chain generation** with all Greeks across multiple strikes
- **Volatility surface construction** with bilinear interpolation and 3D visualization
- **Strategy analysis** with full portfolio Greeks and payoff visualization

For more details, see the [README](../README.md) or explore the `optipricer` package API.